In [ ]:
# | default_exp features.mds_gene

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()

In [ ]:
# | export
class MDSGeneEvaluator(FeatureEvaluator):
    """Gene-specific MDS signatures."""

    name = "MDSGene"
    source_file = ".MDS.gene.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "gene" in cols:
                metrics = [
                    c
                    for c in ["mds_mean", "mds_e1", "mds_std", "mds_z", "mds_e1_z"]
                    if c in cols
                ]
                for row in df.to_dict("records"):
                    g = str(row["gene"]).replace(" ", "_").replace("-", "_")
                    if g == "NAN":
                        continue
                    for m in metrics:
                        if pd.notna(row[m]):
                            extracted[f"{g}_{m}"] = float(row[m])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = MDSGeneEvaluator()
df = pd.DataFrame([{"gene": "TP53", "mds_mean": 0.8, "mds_z": -1.0}])
res = evaluator.extract(df)
assert res["TP53_mds_mean"] == 0.8